# Lab 4 — Build the Audience, Then Buy It
Name: Valentyna Tkachenko | Group: КМ-41

## Setup

In [15]:
!pip install kagglehub -q
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.


In [16]:
import numpy as np

SIX = pd.Timedelta(hours=6)
for df in (programs, viewings):
    for col in ("start_ts", "end_ts"):
        df[col] = pd.to_datetime(df[col]).astype("datetime64[ns]")
programs["broadcast_date"] = pd.to_datetime(programs["broadcast_date"]).astype("datetime64[ns]")
daily_weights["broadcast_date"] = pd.to_datetime(daily_weights["broadcast_date"]).astype("datetime64[ns]")

age = (viewer_attributes[viewer_attributes["attr_id"] == 300]
       .drop_duplicates("respondent_id").set_index("respondent_id")["value"])
sex = (viewer_attributes[viewer_attributes["attr_id"].isin([201, 202])]
       .drop_duplicates("respondent_id").set_index("respondent_id")["attr_id"].map({201: "Men", 202: "Women"}))
people_ids = viewer_attributes.loc[viewer_attributes["attr_id"].isin([101, 102]), "respondent_id"].unique()
adult_ids = set(int(i) for i in people_ids if age.get(i, -1) >= 18)

pw = daily_weights[daily_weights["resp_type"].isin(["R", "L"])].copy()
N_DAYS = pw["broadcast_date"].nunique()

def audience_size(ids):
    return pw.loc[pw["respondent_id"].isin(ids), "weight"].sum() / N_DAYS

def daily_size(ids):
    return pw[pw["respondent_id"].isin(ids)].groupby("broadcast_date")["weight"].sum()

print("Adult panel members (R and L, 18+):", len(adult_ids))
print("Broadcast days:", N_DAYS)

Adult panel members (R and L, 18+): 8010
Broadcast days: 31


## Task 1.0 — Clean the survey file

In [17]:
layout = survey_layout.copy()
layout.columns = [c.strip().lstrip("\ufeff") for c in layout.columns]
answers = layout[layout["RECTYPE"] == "B"].set_index("KEYWORD")
valid_kw = set(answers.index)

raw = survey.copy()
log = []

def note(kind, mask, action):
    log.append({"bad record": kind, "rows": int(mask.sum()), "what I did": action})

is_header = raw["RESPID"].str.strip().eq("RESPID")
is_footer = raw["RESPID"].str.contains("TOTAL", case=False)
note("Header line repeated inside the data", is_header, "dropped")
note("Footer line (TOTAL ROWS)", is_footer, "dropped")
df = raw[~is_header & ~is_footer].copy()
n_data_rows = len(df)

rid = df["RESPID"].str.strip()
note("RESPID with leading/trailing spaces", df["RESPID"] != rid, "stripped the spaces")
note("RESPID written as a decimal (….0)", rid.str.endswith(".0"), "removed .0")
rid = rid.str.replace(r"\.0$", "", regex=True)
note("RESPID with a leading zero (9 digits)", rid.str.match(r"^0\d{8}$"), "removed the leading zero")
df["respondent_id"] = rid.astype(float).astype("int64")

kw = df["KEYWORD"].str.strip()
note("KEYWORD with trailing spaces", df["KEYWORD"] != kw, "stripped the spaces")
note("KEYWORD in lower case (x7…)", kw != kw.str.upper(), "converted to upper case")
df["keyword"] = kw.str.upper()
bad_kw = ~df["keyword"].isin(valid_kw)
note("KEYWORD not in the layout (X880…, X902…)", bad_kw, "dropped")

val = df["VALUE"].str.strip().str.lower()
is_yes = val.isin(["1", "1.0", "y", "yes", "true"])
is_no = val.isin(["0", "0.0", "n", "no", "false"])
note("VALUE yes written another way (1.0, y, Y, TRUE, yes)", ~val.eq("1") & is_yes, "kept as yes")
note("VALUE no (0, 0.0, N, no) in a yes-only file", is_no, "dropped")
note("VALUE empty", val.eq(""), "dropped (not a yes)")
df = df[~bad_kw & is_yes].copy()

not_adult = ~df["respondent_id"].isin(adult_ids)
is_child = not_adult & df["respondent_id"].map(age).lt(18)
note("RESPID of a child (under 18) in the panel", is_child, "dropped")
note("RESPID not in the panel at all", not_adult & ~is_child, "dropped")
df = df[~not_adult]

dup = df.duplicated(["respondent_id", "keyword"])
note("Same person and keyword twice", dup, "kept one")
clean = df[~dup][["respondent_id", "keyword"]]

PAIR_KW = {"X710101": 1, "X710102": 2, "X710103": 3}
pair_rows = clean[clean["keyword"].isin(PAIR_KW)]
two_counts = pair_rows.groupby("respondent_id").size()
note("Person with two pair counts", two_counts.gt(1), "kept the higher count")

people = pd.DataFrame({"respondent_id": sorted(adult_ids)})
pairs = pair_rows.assign(n=pair_rows["keyword"].map(PAIR_KW)).groupby("respondent_id")["n"].max()
people["pairs"] = people["respondent_id"].map(pairs).fillna(0).astype(int)
buyer_ids = set(clean["respondent_id"])
people["buyer"] = people["respondent_id"].isin(buyer_ids)
for kw_, brand in [("X710201", "stride"), ("X710202", "kestrel"), ("X710203", "easywalk")]:
    people[brand] = people["respondent_id"].isin(clean.loc[clean["keyword"] == kw_, "respondent_id"])

print("Survey data rows:", n_data_rows)
print("Clean yes answers:", len(clean))
print("Adult panel members:", len(people))
print("Non-buyers:", int((~people["buyer"]).sum()))
print("People with two pair counts:", int(two_counts.gt(1).sum()))
print("Rows as loaded (incl. 2 repeated headers and 1 footer):", len(raw))
display(pd.DataFrame(log))   # table: kind of bad record, rows, what you did

mine = {"X710100": int(people["buyer"].sum()),
        "X710101": int((people["pairs"] == 1).sum()), "X710102": int((people["pairs"] == 2).sum()),
        "X710103": int((people["pairs"] == 3).sum())}
for k in ["X710201", "X710202", "X710203", "X710298", "X710299"]:
    mine[k] = int((clean["keyword"] == k).sum())
check = pd.DataFrame({"answer": answers.loc[list(mine), "Text"], "my_count": pd.Series(mine),
                      "layout_SAMPLE": answers.loc[list(mine), "SAMPLE"].astype(int)})
check["match"] = check["my_count"] == check["layout_SAMPLE"]
display(check)   # table: KEYWORD, your count, layout SAMPLE

asis = raw[~is_header & ~is_footer]
asis_kestrel = asis.loc[asis["KEYWORD"] == "X710202", "RESPID"].nunique()
asis_buyers = asis.loc[asis["KEYWORD"] == "X710100", "RESPID"].nunique()
print()
print("Kestrel buyers, raw file vs clean:", asis_kestrel, "vs", int(people["kestrel"].sum()),
      "| off by:", round((asis_kestrel / people["kestrel"].sum() - 1) * 100, 1), "%")
print("Share of adults who bought, raw file vs clean:", round(asis_buyers / len(people), 3), "vs",
      round(people["buyer"].mean(), 3))

Survey data rows: 18346
Clean yes answers: 15510
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113
Rows as loaded (incl. 2 repeated headers and 1 footer): 18349


,bad record,rows,what I did
0,Header line repeated inside the data,2,dropped
1,Footer line (TOTAL ROWS),1,dropped
2,RESPID with leading/trailing spaces,573,stripped the spaces
3,RESPID written as a decimal (….0),533,removed .0
4,RESPID with a leading zero (9 digits),349,removed the leading zero
5,KEYWORD with trailing spaces,370,stripped the spaces
6,KEYWORD in lower case (x7…),734,converted to upper case
7,"KEYWORD not in the layout (X880…, X902…)",738,dropped
8,"VALUE yes written another way (1.0, y, Y, TRUE...",1194,kept as yes
9,"VALUE no (0, 0.0, N, no) in a yes-only file",619,dropped


,answer,my_count,layout_SAMPLE,match
X710100,Any,4779,4779,True
X710101,1 pair,1810,1810,True
X710102,2 pairs,1308,1308,True
X710103,3 or more pairs,1661,1661,True
X710201,STRIDE,1563,1563,True
X710202,KESTREL,354,354,True
X710203,EASYWALK,1050,1050,True
X710298,OTHER BRAND,2739,2739,True
X710299,DON'T KNOW,371,371,True



Kestrel buyers, raw file vs clean: 495 vs 354 | off by: 39.8 %
Share of adults who bought, raw file vs clean: 0.548 vs 0.597


**Answer:** The file broke the vendor's promises in many ways. Two header lines were repeated inside the data and a TOTAL ROWS footer was at the end (dropped). RESPIDs came with spaces (573 rows), as decimals ending in .0 (533) or with a leading zero (349): I cleaned them into the panel's 8-digit ids. KEYWORDs had trailing spaces (370) or were in lower case (734): cleaned. 738 rows used codes that are not in the layout (X880…, X902…) and were dropped. The "yes-only" file also held 619 "no" answers (0, N, no) and 123 empty values, which I dropped, while 1,194 yes answers were written another way (1.0, y, TRUE, yes) and kept. 273 rows belonged to children and 150 to ids that are not in the panel (dropped), 933 rows repeated the same person and keyword (kept one), and 113 people had two pair counts (kept the higher). After cleaning, every count matches the layout's SAMPLE exactly (4,779 buyers, 1,810 / 1,308 / 1,661 by pairs, 354 Kestrel buyers).

Taken as delivered, the file would have given 495 Kestrel buyers instead of 354, about 40% too many (duplicates, "no" answers, children and outsiders all counted), and a buyer share of 0.548 instead of 0.597, about 5 points too low (lower-case and space-padded Any codes would be missed, and so would buyers who have a pair count or a brand but no Any row). One file, two errors in opposite directions: that is why the codebook check matters.

## Task 1.1 — Four definitions

In [18]:
people["age"] = people["respondent_id"].map(age)
people["sex"] = people["respondent_id"].map(sex)
AUD = {
    "Adults 18+": set(people["respondent_id"]),
    "Shoe buyers": set(people.loc[people["buyer"], "respondent_id"]),
    "Heavy buyers (3+)": set(people.loc[people["pairs"] == 3, "respondent_id"]),
    "Kestrel buyers": set(people.loc[people["kestrel"], "respondent_id"]),
    "Adults 25-54": set(people.loc[people["age"].between(25, 54), "respondent_id"]),
}
size = {a: audience_size(ids) for a, ids in AUD.items()}

print("Adults 18+ (M):", round(size["Adults 18+"] / 1e6, 1))
print("Shoe buyers (M):", round(size["Shoe buyers"] / 1e6, 1))
print("Heavy buyers, 3+ pairs (M):", round(size["Heavy buyers (3+)"] / 1e6, 1))
print("Kestrel buyers (M):", round(size["Kestrel buyers"] / 1e6, 1))
print("Adults 25-54 (M):", round(size["Adults 25-54"] / 1e6, 1))
print("Kestrel buyers in panel (n):", len(AUD["Kestrel buyers"]))
print()
for a in AUD:
    print(f"{a}: share of adults", round(size[a] / size["Adults 18+"], 3))

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354

Adults 18+: share of adults 1.0
Shoe buyers: share of adults 0.586
Heavy buyers (3+): share of adults 0.196
Kestrel buyers: share of adults 0.04
Adults 25-54: share of adults 0.49


## Task 1.2 — Who are they?

In [19]:
people["age_group"] = pd.cut(people["age"], [17, 34, 54, 200], labels=["18-34", "35-54", "55+"]).astype(str)
people["group"] = people["sex"] + " " + people["age_group"]
GROUPS = [f"{s} {a}" for s in ["Men", "Women"] for a in ["18-34", "35-54", "55+"]]
gid = people.set_index("respondent_id")["group"]

prof = {}
for a, ids in AUD.items():
    w = pw[pw["respondent_id"].isin(ids)]
    by_g = w.groupby(w["respondent_id"].map(gid))["weight"].sum()
    prof[a] = (by_g / by_g.sum()).reindex(GROUPS).fillna(0)
share_tbl = pd.DataFrame(prof)
index_tbl = share_tbl.div(share_tbl["Adults 18+"], axis=0) * 100
profile = pd.concat({"share": share_tbl, "index": index_tbl}, axis=1).swaplevel(axis=1)
profile = profile[[(a, m) for a in AUD for m in ("share", "index")]]
fmt = {c: (3 if c[1] == "share" else 0) for c in profile.columns}
profile.rename_axis("group").round(fmt)   # table: audience x group, share and index vs adults

Adults 18+        Shoe buyers        Heavy buyers (3+)         \
                 share  index       share  index             share  index   
group                                                                       
Men 18-34        0.144  100.0       0.140   97.0             0.111   77.0   
Men 35-54        0.160  100.0       0.158   99.0             0.138   87.0   
Men 55+          0.185  100.0       0.186  101.0             0.189  102.0   
Women 18-34      0.144  100.0       0.140   98.0             0.139   97.0   
Women 35-54      0.160  100.0       0.178  111.0             0.226  142.0   
Women 55+        0.208  100.0       0.197   95.0             0.196   94.0   

            Kestrel buyers        Adults 25-54         
                     share  index        share  index  
group                                                  
Men 18-34            0.118   82.0        0.174  121.0  
Men 35-54            0.189  119.0        0.326  204.0  
Men 55+              0.126   68.0        0.000    0.0  
Women 18-34          0.181  126.0        0.174  121.0  
Women 35-54          0.265  166.0        0.326  204.0  
Women 55+            0.121   58.0        0.000    0.0

**Answer:** **Which definition for Kestrel:** heavy buyers (3+ pairs). It is what the client asked for, it is a real behavioral target (19.6% of adults, 49.6M people) rather than a demographic proxy, and with 1,661 panelists it is big enough for stable program ratings. Kestrel's own buyers are the most precise definition, but only 354 panelists, too few to rate programs reliably (Task 2.3); I would use them as a check, not as the buying target. Adults 25–54, the TV buyer's demo, would aim at the wrong people: 38.5% of heavy buyers are 55+ and fall outside it, while it adds many 25–54s who buy few shoes.

**Why shoe buyers look almost like all adults:** 59% of adults bought athletic shoes in the last year. When a definition covers most of the population, its profile cannot differ much from the population (every index is 95–111); it only becomes distinctive when you narrow it, as heavy buyers (women 35–54, index 142) and Kestrel buyers (women 35–54, 166; 55+, 58–68) show (Lecture 5: coverage vs composition).

**One limit of fused data:** the purchase answers were attached to panelists through their age, sex and income, not measured on the same people. So within a demographic cell a program's viewers buy exactly as often as its non-viewers: the fused attribute cannot show that a program attracts real runners beyond what its demographics predict, and any index it produces just restates the program's demographic mix (Lecture 5, "fusion and lookalikes").

## Task 2.1 — Programs by audience

In [20]:
pv = viewings[viewings["resp_type"].isin(["R", "L"])][["respondent_id", "network_code", "start_ts", "end_ts"]]

def person_minutes(tel):
    t = tel[["telecast_id", "network_code", "broadcast_date", "start_ts", "end_ts"]].rename(
        columns={"start_ts": "t0", "end_ts": "t1"})
    out = []
    for net, tt in t.groupby("network_code"):
        v = pv[pv["network_code"] == net]
        v = v[(v["end_ts"] > tt["t0"].min()) & (v["start_ts"] < tt["t1"].max())]
        x = v.merge(tt, on="network_code")
        x = x[(x["start_ts"] < x["t1"]) & (x["end_ts"] > x["t0"])]
        x["minutes"] = (x["end_ts"].clip(upper=x["t1"]) - x["start_ts"].clip(lower=x["t0"])).dt.total_seconds() / 60
        out.append(x.groupby(["telecast_id", "respondent_id", "broadcast_date"], as_index=False)["minutes"].sum())
    m = pd.concat(out, ignore_index=True)
    return m.merge(pw[["respondent_id", "broadcast_date", "weight"]], on=["respondent_id", "broadcast_date"])

def telecast_ratings(tel, pm, ids):
    t = tel.set_index("telecast_id")
    dur = (t["end_ts"] - t["start_ts"]).dt.total_seconds() / 60
    m = pm[pm["respondent_id"].isin(ids)]
    viewers = (m["weight"] * m["minutes"]).groupby(m["telecast_id"]).sum().reindex(t.index, fill_value=0) / dur
    day_size = t["broadcast_date"].map(daily_size(ids))
    return pd.DataFrame({"viewers": viewers, "rating": viewers / day_size * 100})

def clock_to_min(s):
    h, m = str(s).split(":")
    return int(h) * 60 + int(m) - 360

prog_rows = rate_card[rate_card["program_name"].notna()][["network_code", "program_name", "cost_30s_usd"]]
time_rows = rate_card[rate_card["program_name"].isna()].copy()
time_rows["start_min"] = time_rows["start_time"].map(clock_to_min)
time_rows["end_min"] = time_rows["end_time"].map(clock_to_min)
time_rows = time_rows.explode("weekdays")
time_rows["weekday"] = time_rows["weekdays"].astype(int)

def price_30s(tel):
    t = tel[["telecast_id", "network_code", "program_name", "start_ts"]].merge(
        prog_rows, on=["network_code", "program_name"], how="left")
    shifted = t["start_ts"] - SIX
    t["weekday"] = shifted.dt.weekday
    t["minute"] = shifted.dt.hour * 60 + shifted.dt.minute
    m = t[t["cost_30s_usd"].isna()].drop(columns="cost_30s_usd").merge(
        time_rows[["network_code", "weekday", "start_min", "end_min", "cost_30s_usd"]], on=["network_code", "weekday"])
    m = m[(m["minute"] >= m["start_min"]) & (m["minute"] < m["end_min"])].set_index("telecast_id")["cost_30s_usd"]
    t["cost_30s_usd"] = t["cost_30s_usd"].fillna(t["telecast_id"].map(m))
    return t.set_index("telecast_id")["cost_30s_usd"]

PROGRAMS = [("Sunday Night Football (NBC)", "NBC", "NFL Football"),
            ("Monday Night Football (ABC)", "ABC", "NFL Football"),
            ("Dancing with the Stars (ABC)", "ABC", "Dancing with the Stars"),
            ("NCIS (CBS)", "CBS", "NCIS"),
            ("Hannity (FNC)", "FNC", "Hannity"),
            ("Unsellable Houses (HGTV)", "HGTV", "Unsellable Houses"),
            ("Halloween Wars (FOOD)", "FOOD", "Halloween Wars"),
            ("Grotesquerie (FX)", "FX", "Grotesquerie")]
label = {(n, p): lab for lab, n, p in PROGRAMS}
tel8 = programs[programs["daypart"].eq("Prime")
                & programs.apply(lambda r: (r.network_code, r.program_name) in label, axis=1)].copy()
tel8["program"] = [label[(n, p)] for n, p in zip(tel8["network_code"], tel8["program_name"])]
pm8 = person_minutes(tel8)

HEAVY, ADULTS, KESTREL = AUD["Heavy buyers (3+)"], AUD["Adults 18+"], AUD["Kestrel buyers"]
r_ad = telecast_ratings(tel8, pm8, ADULTS)
r_hv = telecast_ratings(tel8, pm8, HEAVY)
r_ke = telecast_ratings(tel8, pm8, KESTREL)
tel8 = tel8.set_index("telecast_id")
tel8["cost_30s"] = price_30s(tel8.reset_index())
per_tel = pd.DataFrame({"program": tel8["program"], "adults_rating": r_ad["rating"], "adult_viewers": r_ad["viewers"],
                        "target_rating": r_hv["rating"], "target_viewers": r_hv["viewers"],
                        "kestrel_rating": r_ke["rating"], "cost_30s": tel8["cost_30s"]})
prog = per_tel.groupby("program").mean().reindex([p[0] for p in PROGRAMS])
prog["telecasts"] = per_tel.groupby("program").size()
prog["composition"] = prog["target_viewers"] / prog["adult_viewers"]
prog["index"] = prog["target_rating"] / prog["adults_rating"] * 100
prog["adults_cpm"] = prog["cost_30s"] / prog["adult_viewers"] * 1000
prog["target_cpm"] = prog["cost_30s"] / prog["target_viewers"] * 1000
prog["kestrel_index"] = prog["kestrel_rating"] / prog["adults_rating"] * 100

show = prog[["telecasts", "adults_rating", "target_rating", "target_viewers", "composition", "index",
             "cost_30s", "adults_cpm", "target_cpm"]].copy()
show["target_viewers"] = show["target_viewers"] / 1000
show = show.rename(columns={"target_viewers": "target_viewers_k", "cost_30s": "cost_30s_usd",
                            "adults_cpm": "adults_cpm_usd", "target_cpm": "target_cpm_usd"})
display(show.round({"adults_rating": 2, "target_rating": 2, "target_viewers_k": 0, "composition": 3, "index": 0,
                    "cost_30s_usd": 0, "adults_cpm_usd": 2, "target_cpm_usd": 2}).astype({"cost_30s_usd": int}))   # table: 8 programs, the columns of Task 2.1
print("Target universe (M):", round(size["Heavy buyers (3+)"] / 1e6, 1))
print("Heavy buyers' share of adults:", round(size["Heavy buyers (3+)"] / size["Adults 18+"], 3))

,telecasts,adults_rating,target_rating,target_viewers_k,composition,index,cost_30s_usd,adults_cpm_usd,target_cpm_usd
program,,,,,,,,,
Sunday Night Football (NBC),4,5.82,5.79,2865.0,0.195,99.0,487200,33.12,170.05
Monday Night Football (ABC),3,8.93,8.36,4149.0,0.184,94.0,723600,32.09,174.40
Dancing with the Stars (ABC),9,3.25,3.14,1564.0,0.191,97.0,158600,19.33,101.39
NCIS (CBS),3,2.21,2.43,1206.0,0.216,110.0,178800,32.09,148.22
Hannity (FNC),22,1.09,0.84,417.0,0.152,77.0,39900,14.50,95.71
Unsellable Houses (HGTV),13,0.50,0.64,317.0,0.249,127.0,12900,10.12,40.66
Halloween Wars (FOOD),9,0.28,0.33,162.0,0.228,116.0,9700,13.67,59.90
Grotesquerie (FX),25,0.26,0.25,124.0,0.188,96.0,7592,11.50,61.11


Target universe (M): 49.6
Heavy buyers' share of adults: 0.196


## Task 2.2 — Index, volume or price?

In [21]:
print("Highest index (program):", prog["index"].idxmax())
print("Most target viewers (program):", prog["target_viewers"].idxmax())
print("Lowest target CPM (program):", prog["target_cpm"].idxmin())

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** No, they are not the same program. Unsellable Houses (HGTV) has both the highest index (127) and the lowest target CPM (USD 40.66), but it delivers only about 317,000 heavy buyers in an average minute. Monday Night Football (ABC) delivers the most, about 4.1 million, with an index below average (94) and a target CPM of USD 174. Even the best-indexing program has a composition of only 0.249, so three in four of its viewers are not heavy buyers.

To a client who wants "only the highest index" I would say what Lecture 5's shampoo example says: the index measures concentration, not size. A small program with a high index can be a great price per target viewer, but it cannot carry the budget or build reach; you would buy the same few heavy viewers again and again. Big programs with an index near 100 still reach the most target people. The right comparison is the target CPM, with enough volume (reach) for the money, and the index is a tie-breaker, not the goal.

## Task 2.3 — Kestrel buyers

In [22]:
n_tel = per_tel.groupby("program").size()
pm_k = pm8[pm8["respondent_id"].isin(KESTREL)]
pm_h = pm8[pm8["respondent_id"].isin(HEAVY)]
kest = pd.DataFrame({
    "heavy_buyer_index": prog["index"],
    "kestrel_index": prog["kestrel_index"],
    "heavy_buyer_panelists_per_telecast": pm_h.groupby(pm_h["telecast_id"].map(tel8["program"]))["respondent_id"].size() / n_tel,
    "kestrel_panelists_per_telecast": pm_k.groupby(pm_k["telecast_id"].map(tel8["program"]))["respondent_id"].size() / n_tel,
}).reindex(prog.index).fillna(0)
display(kest.round({"heavy_buyer_index": 0, "kestrel_index": 0, "heavy_buyer_panelists_per_telecast": 1,
                    "kestrel_panelists_per_telecast": 1}))   # table: program, heavy-buyer index, Kestrel index

print("Spread of the index across the eight programs (max - min):")
print("  heavy buyers:", round(kest["heavy_buyer_index"].max() - kest["heavy_buyer_index"].min()),
      "| Kestrel buyers:", round(kest["kestrel_index"].max() - kest["kestrel_index"].min()))
print("Panel members: heavy buyers", len(HEAVY), "| Kestrel buyers", len(KESTREL))

,heavy_buyer_index,kestrel_index,heavy_buyer_panelists_per_telecast,kestrel_panelists_per_telecast
program,,,,
Sunday Night Football (NBC),99.0,99.0,329.0,75.2
Monday Night Football (ABC),94.0,86.0,292.7,69.3
Dancing with the Stars (ABC),97.0,83.0,122.6,23.4
NCIS (CBS),110.0,57.0,86.3,11.0
Hannity (FNC),77.0,43.0,15.9,2.1
Unsellable Houses (HGTV),127.0,77.0,25.4,4.0
Halloween Wars (FOOD),116.0,95.0,13.7,2.8
Grotesquerie (FX),96.0,130.0,11.4,3.1


Spread of the index across the eight programs (max - min):
  heavy buyers: 50 | Kestrel buyers: 87
Panel members: heavy buyers 1661 | Kestrel buyers 354


**Answer:** The Kestrel index jumps from 43 (Hannity) to 130 (Grotesquerie), a spread of 87 points against 50 for heavy buyers. The reason is sample size: there are only 354 Kestrel buyers in the panel, against 1,661 heavy buyers. On the cable programs only 2–4 Kestrel panelists watch an average telecast (Hannity 2.1, Grotesquerie 3.1), so one person more or less changes the index by tens of points. Since the purchase data was fused by demographics (Task 1.2), the true differences between programs should be modest; big swings are mostly noise.

I would trust the heavy-buyer index, and for Kestrel only the big programs with dozens of Kestrel panelists per telecast (Sunday Night Football, 75 per telecast: index 99). To check: compute a standard error or bootstrap the panelists to get a confidence interval for each index; split the month into halves and see whether the index repeats; and compare each Kestrel index with the one its demographic mix predicts (the program's age/sex/income mix × Kestrel's index in each group). If the measured index is far from the predicted one, it is noise.

## Task 3.1 — Weights

In [23]:
def to_eastern(s):
    return pd.to_datetime(s, utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)

acr = acr_sessions
acr["start_et"] = to_eastern(acr["start_utc"])
acr["end_et"] = to_eastern(acr["end_utc"])
acr["broadcast_date"] = (acr["start_et"] - SIX).dt.normalize().astype("datetime64[ns]")
kept = acr[(acr["end_et"] - acr["start_et"]).dt.total_seconds() >= 60]
intab = kept[["tv_id", "broadcast_date"]].drop_duplicates()

tv_ip = acr.drop_duplicates("tv_id")[["tv_id", "hashed_ip"]]
hh_per_ip = hh_attributes.groupby("hashed_ip")["household_id"].nunique()
one_ip = hh_per_ip[hh_per_ip == 1].index
matched = tv_ip[tv_ip["hashed_ip"].isin(one_ip)].merge(
    hh_attributes[["hashed_ip", "head_age_band", "income_band"]], on="hashed_ip")
matched["head_age"] = np.where(matched["head_age_band"] == "65+", "65+", "Under 65")

m_intab = intab.merge(matched[["tv_id", "head_age", "income_band"]], on="tv_id")
cell_n = m_intab.groupby(["broadcast_date", "head_age", "income_band"]).size().rename("tvs").reset_index()
cell_n = cell_n.merge(universe, on=["head_age", "income_band"])
cell_n["weight"] = cell_n["tv_households"] / cell_n["tvs"]
m_intab = m_intab.merge(cell_n[["broadcast_date", "head_age", "income_band", "weight"]],
                        on=["broadcast_date", "head_age", "income_band"])
w_sum = m_intab.groupby("broadcast_date")["weight"].sum() / 1e6

print("TVs matched to one household:", len(matched))
print("Weights sum per day (M):", round(w_sum.min(), 1), "to", round(w_sum.max(), 1))
print("Cells:", cell_n[["head_age", "income_band"]].drop_duplicates().shape[0], "| universe (M):",
      round(universe["tv_households"].sum() / 1e6, 1))

live = kept[kept["session_type"] == "live"]
ref = reference_ratings.merge(programs[["telecast_id", "network_code", "program_name", "broadcast_date",
                                        "start_ts", "end_ts"]], on="telecast_id")
rows = []
for t in ref.itertuples():
    s = live[(live["network_code"] == t.network_code) & (live["start_et"] < t.end_ts) & (live["end_et"] > t.start_ts)]
    ov = (s["end_et"].clip(upper=t.end_ts) - s["start_et"].clip(lower=t.start_ts)).dt.total_seconds() / 60
    watched = set(ov.groupby(s["tv_id"]).sum().loc[lambda x: x >= 1].index)
    day = m_intab[m_intab["broadcast_date"] == t.broadcast_date]
    saw = day["tv_id"].isin(watched)
    rows.append({"telecast": f"{t.program_name} ({t.network_code})", "date": t.broadcast_date.strftime("%Y-%m-%d"),
                 "unweighted": saw.mean() * 100,
                 "weighted": day.loc[saw, "weight"].sum() / day["weight"].sum() * 100,
                 "reference": t.hh_rating})
wt = pd.DataFrame(rows)
wt["moved"] = wt["weighted"] - wt["unweighted"]
display(wt.round({"unweighted": 2, "weighted": 2, "reference": 2, "moved": 2}))   # table: telecast, unweighted, weighted, reference
gap_u = (wt["reference"] - wt["unweighted"]).abs().mean()
gap_w = (wt["reference"] - wt["weighted"]).abs().mean()
print("Mean gap unweighted (points):", round(gap_u, 2))
print("Mean gap weighted (points):", round(gap_w, 2))
print("Share of gap closed:", round(1 - gap_w / gap_u, 3))

TVs matched to one household: 19607
Weights sum per day (M): 125.8 to 125.8
Cells: 12 | universe (M): 125.8


,telecast,date,unweighted,weighted,reference,moved
0,MLB Postseason (FOX),2024-10-25,17.03,17.76,25.86,0.72
1,NFL Football (NBC),2024-10-27,22.62,23.75,33.14,1.13
2,NCIS (CBS),2024-10-14,5.50,5.80,10.56,0.29
3,Hannity (FNC),2024-10-16,1.04,1.35,2.30,0.31
4,Unsellable Houses (HGTV),2024-10-16,2.12,2.08,3.69,-0.04
5,Halloween Baking Championship (FOOD),2024-10-14,0.95,0.90,1.63,-0.05
6,Grotesquerie (FX),2024-10-16,2.17,2.11,3.53,-0.06
7,The Real Housewives of New York City (BRAVO),2024-10-15,0.71,0.64,1.23,-0.07


Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


In [24]:
mix = pd.DataFrame({
    "matched_TVs": matched.groupby(["head_age", "income_band"]).size() / len(matched),
    "universe": universe.set_index(["head_age", "income_band"])["tv_households"] / universe["tv_households"].sum()})
mix["index"] = mix["matched_TVs"] / mix["universe"] * 100
mix.round(3)

matched_TVs  universe    index
head_age income_band                                
65+      $100k-$150k        0.035     0.039   90.744
         $150k+             0.037     0.040   93.001
         $25k-$50k          0.033     0.062   53.858
         $50k-$75k          0.031     0.048   64.594
         $75k-$100k         0.029     0.034   85.755
         Under $25k         0.028     0.060   46.146
Under 65 $100k-$150k        0.176     0.139  127.105
         $150k+             0.228     0.194  117.461
         $25k-$50k          0.088     0.100   88.073
         $50k-$75k          0.115     0.108  105.928
         $75k-$100k         0.127     0.092  137.961
         Under $25k         0.072     0.084   86.032

## Task 3.2 — What weighting cannot fix

**Answer:** **What moved:** the cell weights add up to 125.8M TV households every day, and they moved the older-skewing programs up: Sunday Night Football (NBC) +1.13 points, the World Series (FOX) +0.72, Hannity (FNC) +0.31 and NCIS (CBS) +0.29. The young-skewing cable programs moved slightly down: HGTV, Food Network, FX and Bravo by −0.04 to −0.07. The footprint has far too few homes with a head aged 65+, especially lower-income ones (indexes 46–93 in the 65+ cells, against 86–138 under 65), so weighting gives those homes more weight, and they watch more broadcast and Fox News.

**What is left:** weighting closed only 7.5% of the mean gap (3.73 → 3.44 points). It can only fix the mix on the variables in the cells. It cannot fix that each home is measured on one TV while the reference counts every set (about 2.6 per home, Lecture 4); live viewing that ACR does not recognize ("unknown"); that there are no people in the data; and that the 35% of TVs with no single-household match, and the homes that opted out or never bought this brand, may watch differently even within the same cell.

**Why income barely matters:** viewing of these programs depends mainly on age, not on income. Within an age group, richer and poorer homes watch these programs at similar rates, so re-balancing income moves the ratings very little; and the cells split age only at 65, so the age effect within "Under 65" is not corrected at all.

## Task 4.1 — October predicts October

In [25]:
BIG5 = ["CBS", "NBC", "ABC", "FOX", "FNC"]
H0, H1 = pd.Timestamp("2024-10-01"), pd.Timestamp("2024-10-21")
F0, F1 = pd.Timestamp("2024-10-22"), pd.Timestamp("2024-10-28")
prime = programs[(programs["daypart"] == "Prime") & programs["network_code"].isin(BIG5)].copy()
hist = prime[prime["broadcast_date"].between(H0, H1)]
fc = prime[prime["broadcast_date"].between(F0, F1)
           & prime.set_index(["network_code", "program_name"]).index.isin(
               hist.set_index(["network_code", "program_name"]).index)]
both = pd.concat([hist, fc])
pm_all = person_minutes(both)
ALL_PERSONS = set(pw["respondent_id"])

res = []
for aud, ids in [("P2+", ALL_PERSONS), ("Heavy buyers", HEAVY)]:
    r = telecast_ratings(both, pm_all, ids)["rating"]
    h = hist.assign(rating=hist["telecast_id"].map(r)).groupby(["network_code", "program_name"])["rating"].mean()
    f = fc.assign(actual=fc["telecast_id"].map(r))
    f["forecast"] = [h[(n, p)] for n, p in zip(f["network_code"], f["program_name"])]
    f["audience"] = aud
    res.append(f)
res = pd.concat(res)
res = res[res["actual"] > 0].copy()
res["ape"] = (res["actual"] - res["forecast"]).abs() / res["actual"]
mape = res.groupby("audience")["ape"].mean() * 100
p2 = res[res["audience"] == "P2+"].sort_values("ape", ascending=False)
p2["program"] = p2["program_name"] + " (" + p2["network_code"] + ")"
worst = p2.iloc[0]

print("Telecasts forecast (n):", len(fc))
print("MAPE, P2+ (%):", round(mape["P2+"], 1))
print("MAPE, heavy buyers (%):", round(mape["Heavy buyers"], 1))
print("Biggest miss, P2+ (program, date):", worst["program"], worst["broadcast_date"].strftime("%Y-%m-%d"))
print()
print("Telecasts skipped for an actual of 0: P2+", int(len(fc) - (res["audience"] == "P2+").sum()),
      "| heavy buyers", int(len(fc) - (res["audience"] == "Heavy buyers").sum()))
print("WAPE, P2+ (%):", round((p2["actual"] - p2["forecast"]).abs().sum() / p2["actual"].sum() * 100, 1))

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX) 2024-10-27

Telecasts skipped for an actual of 0: P2+ 0 | heavy buyers 0
WAPE, P2+ (%): 22.8


## Task 4.2 — Why it missed

In [26]:
top5 = p2.head(5).assign(date=lambda t: t["broadcast_date"].dt.strftime("%Y-%m-%d"),
                         start=lambda t: t["start_ts"].dt.strftime("%H:%M"))
display(top5[["program", "date", "start", "forecast", "actual", "ape"]]
        .round({"forecast": 2, "actual": 2, "ape": 3}).reset_index(drop=True))   # table: the five biggest misses (program, date, forecast, actual)

nights = top5[["network_code", "broadcast_date"]].drop_duplicates()
ctx = programs.merge(nights, on=["network_code", "broadcast_date"])
ctx = ctx[ctx["start_ts"].dt.hour.between(17, 23)]
ctx.assign(date=ctx["broadcast_date"].dt.strftime("%Y-%m-%d"),
           start=ctx["start_ts"].dt.strftime("%H:%M"), end=ctx["end_ts"].dt.strftime("%H:%M"))[
    ["network_code", "date", "program_name", "episode_name", "start", "end"]].reset_index(drop=True)

,program,date,start,forecast,actual,ape
0,Krapopolis (FOX),2024-10-27,21:30,1.06,0.22,3.878
1,Universal Basic Guys (FOX),2024-10-27,20:30,0.68,0.26,1.622
2,Fire Country (CBS),2024-10-27,22:44,1.70,0.67,1.548
3,Accused (FOX),2024-10-22,21:00,0.95,0.37,1.546
4,What Would You Do? (ABC),2024-10-23,22:00,0.44,0.18,1.381


,network_code,date,program_name,episode_name,start,end
0,ABC,2024-10-23,ABC World News Tonight with David Muir,Episode 285,18:30,19:00
1,ABC,2024-10-23,Local / Syndicated Programming,None,19:00,20:00
2,ABC,2024-10-23,The Golden Bachelorette,Week 6,20:00,21:30
3,ABC,2024-10-23,Abbott Elementary,Class Pet,21:30,22:00
4,ABC,2024-10-23,What Would You Do?,Episode 11,22:00,23:00
5,ABC,2024-10-23,Local / Syndicated Programming,None,23:00,00:35
6,CBS,2024-10-27,60 Minutes,Deportation | Sanctions | Surfmen,19:35,20:00
7,CBS,2024-10-27,Poppa's House,None,20:00,20:30
8,CBS,2024-10-27,Fire Country,None,20:30,20:44
9,CBS,2024-10-27,Tracker,Bloodlines,20:44,21:44


**Answer:** All five biggest misses were forecasts that were too high, and four of them were schedule effects. On Sunday Oct 27, Krapopolis (FOX) was forecast at 1.06 and got 0.22, and Universal Basic Guys (FOX) at 0.68 and got 0.26. Their history means mixed different slots: Krapopolis rated 1.8–1.9 on Wednesdays at 22:00 but only 0.26–0.30 on Sundays at 21:30, and Universal Basic Guys got 1.09 on Oct 20 right after the NFL lead-in, while on Oct 27 it aired at 20:30. Fire Country (CBS) on Oct 27 was forecast at 1.70 and got 0.67: CBS's NFL game ran long (Bears at Commanders until 19:35), the evening shifted, and the "telecast" was a 16-minute tail at 22:44 after The Equalizer, far from its usual Friday 21:00 slot. Accused (FOX, Oct 22) and What Would You Do? (ABC, Oct 23) were declining week by week, and the history mean still carried earlier, stronger weeks (Accused 1.87 on Oct 1).

The heavy-buyer forecast is less accurate (MAPE 38.6% vs 35.8%) because it rests on far fewer panelists: 1,661 heavy buyers against about 9,300 persons in-tab per day. Both the history and the actual week are noisier, and the error grows as the in-target sample shrinks (Lecture 6, slide 11).

Two things a better forecast would use: (1) the schedule: day of week, time slot and lead-in (including sports overruns and specials), by forecasting each program from its own slot's history rather than one mean; and (2) recency: weight recent weeks more (or fit a trend) and flag premieres vs repeats. For the target audience it would also help to forecast a big audience (P2+ or adults) and apply a smoothed composition instead of using the noisy target rating directly.

## Task 5.1 — The plan

In [27]:
import itertools

BUDGET = 1_000_000
weekly = (prog["telecasts"] * 7 / 31).round().clip(lower=1).astype(int)
cand = prog[["cost_30s", "target_viewers", "adult_viewers", "target_cpm", "index"]].copy()
cand["max_spots"] = weekly * 2
cand = cand.drop(index="Halloween Wars (FOOD)")
cand["cost_30s"] = cand["cost_30s"].round()

best = None
for n in itertools.product(*[range(m + 1) for m in cand["max_spots"]]):
    n = np.array(n)
    cost = (n * cand["cost_30s"].values).sum()
    if cost <= BUDGET:
        imps = (n * cand["target_viewers"].values).sum()
        if best is None or imps > best[0]:
            best = (imps, cost, n)

plan = cand.assign(spots=best[2])
plan = plan[plan["spots"] > 0]
plan["cost_usd"] = plan["spots"] * plan["cost_30s"]
plan["target_impressions_m"] = plan["spots"] * plan["target_viewers"] / 1e6
plan["adult_impressions_m"] = plan["spots"] * plan["adult_viewers"] / 1e6
plan = plan.sort_values("target_cpm")
display(plan[["spots", "max_spots", "cost_30s", "cost_usd", "target_impressions_m", "target_cpm"]]
        .round({"target_impressions_m": 2, "target_cpm": 2})
        .astype({"cost_30s": int, "cost_usd": int}))   # table: program, spots, cost, target impressions
print("Plan cost ($):", int(plan["cost_usd"].sum()))
print("Plan target impressions (M):", round(plan["target_impressions_m"].sum(), 1))
print()
print("Plan target CPM ($):", round(plan["cost_usd"].sum() / plan["target_impressions_m"].sum() / 1000, 2))
print("Plan composition (target / adult impressions):",
      round(plan["target_impressions_m"].sum() / plan["adult_impressions_m"].sum(), 3))
print("Target GRPs (target impressions / target universe x 100):",
      round(plan["target_impressions_m"].sum() * 1e6 / size["Heavy buyers (3+)"] * 100, 1))
mape_h = mape["Heavy buyers"] / 100
print("Delivery range with the heavy-buyer MAPE (M):",
      round(plan["target_impressions_m"].sum() * (1 - mape_h), 1), "to",
      round(plan["target_impressions_m"].sum() * (1 + mape_h), 1))

,spots,max_spots,cost_30s,cost_usd,target_impressions_m,target_cpm
program,,,,,,
Unsellable Houses (HGTV),6,6,12900,77400,1.90,40.66
Grotesquerie (FX),11,12,7592,83512,1.37,61.11
Hannity (FNC),9,10,39900,359100,3.75,95.71
Dancing with the Stars (ABC),3,4,158600,475800,4.69,101.39


Plan cost ($): 995812
Plan target impressions (M): 11.7

Plan target CPM ($): 85.01
Plan composition (target / adult impressions): 0.182
Target GRPs (target impressions / target universe x 100): 23.6
Delivery range with the heavy-buyer MAPE (M): 7.2 to 16.2


## Task 5.2 — Recommendation

**Answer:** I recommend buying 6 spots in Unsellable Houses (HGTV), 11 in Grotesquerie (FX), 9 in Hannity (FNC) and 3 in Dancing with the Stars (ABC), for USD 995,812 and about 11.7M heavy-buyer impressions (23.6 target GRPs). I chose the programs by target CPM within each program's weekly availability (two spots per telecast), not by index alone: HGTV has both the highest index and the lowest target CPM, FX and Hannity are cheap per heavy buyer even though Hannity's index is only 77, and Dancing with the Stars adds volume. The plan's target CPM is USD 85.01, half that of the NFL (USD 170–174), and Halloween Wars is left out because it will not air in November. The delivery is not certain: with the Task 4 heavy-buyer error of about 39%, the week could plausibly deliver anywhere from about 7M to 16M target impressions, and the cable estimates rest on few panelists, so I would ask the sellers for an audience guarantee with make-goods. The plan's composition is 0.182, so about 82% of the impressions, and of the money, reaches people who are not heavy buyers; that is unavoidable on TV for a target that is a fifth of adults. After the week I would compare delivered heavy-buyer and adult impressions with the plan program by program (and claim make-goods where they fall short), measure reach and frequency among heavy buyers, and check whether Kestrel sales or brand searches rose in exposed homes against unexposed ones.